In [2]:
from email import generator

import torch

torch.__version__

'2.14.0+cu130'

In [23]:
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

In [32]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE == "cuda"
NUM_WORKER = 0 if PIN_MEMORY else 0

SEED=2026
generator = torch.Generator(device=DEVICE)
np.random.seed(SEED)

In [33]:
# 데이터 다운로드
transform = transforms.Compose([
    transforms.ToTensor()
])

train_raw = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=None
)

test_raw = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=None
)

In [34]:
print(len(train_raw))
print(len(test_raw))

print()

print(transforms.ToTensor()(train_raw[0][0]).shape)

50000
10000

torch.Size([3, 32, 32])


In [35]:
class CustomCIFAR10(Dataset):

    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    # idx를 이용해서 데이터 뽑아주기
    def __getitem__(self, idx):

        image, label = self.dataset[idx]

        # transform이 존재한다면 반환해주기
        if self.transform:
            image = self.transform(image)

        return image, label

In [37]:
train_dataset = CustomCIFAR10(
    train_raw,
    transform=transform
)

test_dataset = CustomCIFAR10(
    test_raw,
    transform=transform
)

In [30]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    generator=generator,
    pin_memory=PIN_MEMORY,
    num_workers=NUM_WORKER
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    generator=generator,
    pin_memory=PIN_MEMORY,
    num_workers=NUM_WORKER
)

In [31]:
print(next(iter(test_loader)))

RuntimeError: Expected a 'cpu' device type for generator but found 'cuda'